## Billiard Ball Tracking and Collision Detection Pipeline

This script implements a robust pipeline for tracking billiard balls in a 2-ball pool game video, detecting collisions, and generating an annotated output video.

### 1. Setup and Dependency Installation

In [8]:
# Install necessary libraries
!pip install -qq ultralytics yt-dlp opencv-python

# Ensure ffmpeg is available (usually pre-installed in Colab)
# If not, you might need to install it: !apt-get update && apt-get install -y ffmpeg
print("Dependencies installed: ultralytics, yt-dlp, opencv-python.")
print("FFmpeg is usually pre-installed in Colab environment.")

Dependencies installed: ultralytics, yt-dlp, opencv-python.
FFmpeg is usually pre-installed in Colab environment.


### 2. Ingestion & Codec Normalization

This section downloads the target video from YouTube and normalizes its codec using FFmpeg to ensure compatibility with OpenCV.

In [9]:
import os
import subprocess

# Video URL and file paths
VIDEO_URL = "https://www.youtube.com/watch?v=jx3s7mlYcCU"
RAW_VIDEO_PATH = "raw_download.mp4"
RAW_VIDEO_WEBM_PATH = "raw_download.mp4.webm" # yt-dlp sometimes downloads as .webm
STANDARD_VIDEO_PATH = "standard_input.mp4"

# Download the video using yt-dlp
print(f"Downloading video from {VIDEO_URL}...")
subprocess.run(['yt-dlp', '-o', RAW_VIDEO_PATH, VIDEO_URL], check=True)
print(f"Video downloaded to {RAW_VIDEO_PATH}.")

# Check if yt-dlp saved it as .webm and rename if necessary
if os.path.exists(RAW_VIDEO_WEBM_PATH) and not os.path.exists(RAW_VIDEO_PATH):
    os.rename(RAW_VIDEO_WEBM_PATH, RAW_VIDEO_PATH)
    print(f"Renamed {RAW_VIDEO_WEBM_PATH} to {RAW_VIDEO_PATH}.")

# Normalize video codec with FFMPEG
print(f"Normalizing video codec to H.264 and yuv420p format for {RAW_VIDEO_PATH}...")
ffmpeg_command = [
    'ffmpeg',
    '-i', RAW_VIDEO_PATH,
    '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p',
    '-y', # Overwrite output files without asking
    STANDARD_VIDEO_PATH
]
subprocess.run(ffmpeg_command, check=True)
print(f"Normalized video saved as {STANDARD_VIDEO_PATH}.")

# Clean up raw download
os.remove(RAW_VIDEO_PATH)
print(f"Removed raw downloaded video {RAW_VIDEO_PATH}.")

Video downloaded to raw_download.mp4.
Renamed raw_download.mp4.webm to raw_download.mp4.
Normalizing video codec to H.264 and yuv420p format for raw_download.mp4...
Normalized video saved as standard_input.mp4.
Removed raw downloaded video raw_download.mp4.


### 3. Object Detection, Tracking, and Collision Pipeline

This is the core of the script, handling YOLOv8s tracking, spatial coordinate harvesting, pairwise geometric evaluation with dynamic thresholding, and temporal cooldown for collision detection. It also manages frame annotation and logging.

In [10]:
from ultralytics import YOLO
import cv2
import numpy as np
import math
import os
import shutil

# --- Configuration Parameters ---
MODEL_NAME = 'yolov8s.pt'
TARGET_CLASS_ID = 32  # COCO Class Index for 'sports ball'
CONF_THRESHOLD = 0.10
IOU_THRESHOLD = 0.50
COOLDOWN_FRAMES = 15
COLLISION_BUFFER_FACTOR = 1.15 # Factor for dynamic collision threshold (1.0 = exact touch, 1.15 = slight overlap)
OUTPUT_FRAMES_DIR = 'frames_dir'

# --- Setup Output Directory ---
if os.path.exists(OUTPUT_FRAMES_DIR):
    shutil.rmtree(OUTPUT_FRAMES_DIR)
os.makedirs(OUTPUT_FRAMES_DIR)
print(f"Output directory '{OUTPUT_FRAMES_DIR}' created/cleared.")

# --- Load YOLOv8s Model ---
model = YOLO(MODEL_NAME)
print(f"YOLOv8s model '{MODEL_NAME}' loaded.")

# --- Initialize Video Capture ---
cap = cv2.VideoCapture(STANDARD_VIDEO_PATH)
if not cap.isOpened():
    raise IOError(f"Cannot open video file: {STANDARD_VIDEO_PATH}")

FRAME_WIDTH = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
FRAME_HEIGHT = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS = cap.get(cv2.CAP_PROP_FPS)
TOTAL_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Processing video: {STANDARD_VIDEO_PATH}")
print(f"Resolution: {FRAME_WIDTH}x{FRAME_HEIGHT}, FPS: {FPS}, Total Frames: {TOTAL_FRAMES}")

# --- Main Tracking and Collision Logic Variables ---
collision_score = 0
cooldown_timer = 0
frame_index = 0
collision_banner_frames = 0 # Counter for showing collision banner

print("Starting frame processing loop...")
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_index += 1

    # --- Object Detection & Tracking ---
    # Perform tracking with specified hyperparameters
    # Filter detections strictly to COCO Class Index 32 ('sports ball').
    # Set Confidence Threshold to 0.10 (`conf=0.10`)
    # Set Intersection over Union to 0.50 (`iou=0.50`)
    results = model.track(
        frame,
        persist=True,
        classes=TARGET_CLASS_ID,
        conf=CONF_THRESHOLD,
        iou=IOU_THRESHOLD,
        verbose=False
    )

    # Extract detection and tracking data
    detections = []
    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu().numpy().astype(int) # xywh format
        track_ids = results[0].boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box

            # --- Spatial Coordinate Harvesting ---
            # Compute precise integer pixel centroid coordinates
            cx = int(x)
            cy = int(y)
            # Compute operational diameter tracking scale
            d = max(w, h)

            detections.append({'id': track_id, 'cx': cx, 'cy': cy, 'd': d, 'box': (x-w//2, y-h//2, x+w//2, y+h//2)})

            # Draw bounding box and center dot on frame
            x1, y1, x2, y2 = x - w // 2, y - h // 2, x + w // 2, y + h // 2
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2) # Red bounding box
            cv2.circle(frame, (cx, cy), 5, (0, 255, 255), -1) # Yellow center dot
            cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    # --- Temporal Cooldown State Machine (Multi-Count Prevention) ---
    if cooldown_timer > 0:
        cooldown_timer -= 1
        if collision_banner_frames > 0:
            collision_banner_frames -= 1
        # Bypass spatial math during cooldown
    else:
        # --- Pairwise Geometric Evaluation & Dynamic Thresholding ---
        if len(detections) >= 2:
            # Assuming only two balls for 2-ball pool
            ball1 = detections[0]
            ball2 = detections[1]

            cx1, cy1, d1 = ball1['cx'], ball1['cy'], ball1['d']
            cx2, cy2, d2 = ball2['cx'], ball2['cy'], ball2['d']

            # Calculate Euclidean distance between centroids
            distance = math.sqrt((cx2 - cx1)**2 + (cy2 - cy1)**2)

            # Calculate Scale-Agnostic Adaptive Threshold
            average_diameter = (d1 + d2) / 2
            collision_threshold = average_diameter * COLLISION_BUFFER_FACTOR

            # --- Collision Validation ---
            if distance < collision_threshold:
                # Collision detected and not in cooldown
                collision_score += 1
                cooldown_timer = COOLDOWN_FRAMES # Lock cooldown
                collision_banner_frames = 12 # Show banner for 12 frames

                # --- Live Terminal Logging ---
                print(f"💥 COLLISION DETECTED | Frame Index: {frame_index} | Current Score: {collision_score}")

    # --- Collision Alert Banner (File Archival & Live Terminal Logging) ---
    if collision_banner_frames > 0:
        banner_text = f"CRITERIA MET: COLLISION EVENT #{collision_score}"
        text_size = cv2.getTextSize(banner_text, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
        text_x = (FRAME_WIDTH - text_size[0]) // 2
        text_y = 50 # Position banner at the top

        # Draw a solid green rectangle as a background for the banner
        cv2.rectangle(frame, (0, 0), (FRAME_WIDTH, 80), (0, 255, 0), -1) # Green background
        cv2.putText(frame, banner_text, (text_x, text_y), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 3) # Black text

    # Save every annotated frame image sequentially
    output_frame_path = os.path.join(OUTPUT_FRAMES_DIR, f"frame_{frame_index:05d}.jpg")
    cv2.imwrite(output_frame_path, frame)

    # Optional: Display frame (can be slow in Colab if not using `cv2_imshow`)
    # If you want to see frames live, uncomment and ensure `from google.colab.patches import cv2_imshow`
    # cv2_imshow(frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

print(f"Finished processing {frame_index} frames. Total Collisions: {collision_score}")

cap.release()
# cv2.destroyAllWindows() # Not needed if not using interactive display
print("Video processing complete.")

Output directory 'frames_dir' created/cleared.
YOLOv8s model 'yolov8s.pt' loaded.
Processing video: standard_input.mp4
Resolution: 720x1280, FPS: 30.0, Total Frames: 501
Starting frame processing loop...
💥 COLLISION DETECTED | Frame Index: 415 | Current Score: 1
Finished processing 501 frames. Total Collisions: 1
Video processing complete.


### 4. Final Video Packaging

This section uses FFmpeg to compile the saved annotated frames back into a single output video file.

In [11]:
import subprocess
import os

FINAL_OUTPUT_VIDEO = "final_submission.mp4"

print(f"Compiling annotated frames from '{OUTPUT_FRAMES_DIR}/' into '{FINAL_OUTPUT_VIDEO}'...")

# Get the original FPS to use for output video
# Re-open the original video to get FPS if cap.release() was called
cap_info = cv2.VideoCapture(STANDARD_VIDEO_PATH)
original_fps = cap_info.get(cv2.CAP_PROP_FPS)
cap_info.release()

ffmpeg_compile_command = [
    'ffmpeg',
    '-framerate', str(original_fps), # Use original framerate
    '-i', f'{OUTPUT_FRAMES_DIR}/frame_%05d.jpg',
    '-c:v', 'libx264',
    '-pix_fmt', 'yuv420p',
    '-y', # Overwrite output files without asking
    FINAL_OUTPUT_VIDEO
]
subprocess.run(ffmpeg_compile_command, check=True)

print(f"Final annotated video saved as {FINAL_OUTPUT_VIDEO}")
print("Pipeline execution complete!")

Compiling annotated frames from 'frames_dir/' into 'final_submission.mp4'...
Final annotated video saved as final_submission.mp4
Pipeline execution complete!


### 5. Project Documentation (README.md)

Copy the content below into a file named `README.md` for your GitHub repository.

```markdown
# 🎱 Billiard Ball Tracking & Collision Detection

A self-contained Computer Vision pipeline designed to track pool balls and detect collisions in video feeds using YOLOv8 and geometric spatial analysis.

## 🚀 Project Overview
This project automates the detection of 'impact events' between billiard balls. It handles everything from raw video ingestion to the production of a frame-annotated technical demonstration.

## 🛠️ Technical Stack
*   **Language:** Python 3.x
*   **Model:** YOLOv8s (Ultralytics)
*   **Inference:** COCO Class 32 (Sports Ball)
*   **Video Engineering:** FFmpeg (Codec Normalization H.264/YUV420p)
*   **Image Processing:** OpenCV (Computer Vision)

## 🧠 Key Features
1.  **Normalization Pipeline:** Uses FFmpeg to ensure input videos are compatible with OpenCV's capture properties regardless of original codec.
2.  **Adaptive Thresholding:** Collision detection is not hard-coded. The system calculates a dynamic threshold based on 1.15x the average diameter of the tracked objects to account for camera perspective and scale.
3.  **Temporal Cooldown State Machine:** Implemented a 15-frame cooldown timer to prevent 'double-counting' of a single collision event.
4.  **Automated Logging:** Real-time terminal logs for every detected event with corresponding frame indices.

## 📊 Results
*   **Video Length:** 501 Frames
*   **Target:** 2-Ball Pool Game
*   **Accuracy:** Successfully detected 1 collision at Frame 415.
*   **Output:** `final_submission.mp4` with visual overlays (Bounding Boxes, Centroids, and Collision Banners).

## 📥 How to Use
1. Install dependencies: `pip install ultralytics yt-dlp opencv-python`
2. Update the `VIDEO_URL` in the script.
3. Run the pipeline to generate the `final_submission.mp4` file.
```